# Annotation Remapping Walkthrough

Step-by-step validation of the M3 remapping workflow using the pilot slide `CHN_AU_10_19-21`. The notebook inspects the source GeoJSON, runs the remap module inline, reviews the report, and overlays cut-local annotations on low-resolution cut TIFFs one cut at a time.


## 1. Setup

This notebook expects the pilot cut TIFFs and `{stem}_cuts.json` manifest to exist under `data/cuts/CHN_AU_10_19-21/`. It reruns remapping inline so the intermediate report and output files are visible in the notebook.


In [ ]:
from __future__ import annotations

import json
import sys
from collections import Counter
from pathlib import Path

import matplotlib.pyplot as plt
import pandas as pd
import tifffile
from IPython.display import display

NOTEBOOK_CWD = Path.cwd().resolve()
REPO_ROOT = NOTEBOOK_CWD if (NOTEBOOK_CWD / "data").exists() else NOTEBOOK_CWD.parent
assert (REPO_ROOT / "data").exists(), REPO_ROOT
if str(REPO_ROOT) not in sys.path:
    sys.path.insert(0, str(REPO_ROOT))

from src.data_preparation.remap_annotations import remap_annotations

STEM = "CHN_AU_10_19-21"
DATASET_DIR = REPO_ROOT / "data" / "dataset_28_04"
CUTS_DIR = REPO_ROOT / "data" / "cuts" / STEM
GEOJSON_PATH = DATASET_DIR / f"{STEM}.geojson"
MANIFEST_PATH = CUTS_DIR / f"{STEM}_cuts.json"
REPORT_PATH = CUTS_DIR / f"{STEM}_remap_report.json"
STAGE_COLOURS = {
    0: "steelblue",
    1: "gold",
    2: "darkorange",
    3: "tomato",
    4: "mediumorchid",
}

assert GEOJSON_PATH.exists(), GEOJSON_PATH
assert MANIFEST_PATH.exists(), MANIFEST_PATH


## 2. Inspect the source annotations


In [ ]:
source_geojson = json.loads(GEOJSON_PATH.read_text())
source_features = source_geojson["features"]

geometry_counts = Counter(feature["geometry"]["type"] for feature in source_features)
stage_counts = Counter(
    feature.get("properties", {}).get("metadata", {}).get("ANNOTATION_DESCRIPTION", "Unknown")
    for feature in source_features
)

summary_df = pd.DataFrame(
    {
        "metric": ["n_features", "geometry_types", "stage_labels"],
        "value": [
            len(source_features),
            dict(geometry_counts),
            dict(stage_counts),
        ],
    }
)
display(summary_df)

source_features[:2]


## 3. Inspect the cut manifest


In [ ]:
manifest = json.loads(MANIFEST_PATH.read_text())
cuts_df = pd.DataFrame(
    [
        {
            "cut_index": cut["index"],
            "cut_name": cut["name"],
            "x0": cut["level0_bbox"]["x0"],
            "y0": cut["level0_bbox"]["y0"],
            "x1": cut["level0_bbox"]["x1"],
            "y1": cut["level0_bbox"]["y1"],
            "width": cut["level0_bbox"]["x1"] - cut["level0_bbox"]["x0"],
            "height": cut["level0_bbox"]["y1"] - cut["level0_bbox"]["y0"],
        }
        for cut in manifest["cuts"]
    ]
)
display(cuts_df)


## 4. Run remapping inline


In [ ]:
report = remap_annotations(
    GEOJSON_PATH,
    MANIFEST_PATH,
    CUTS_DIR,
    skip_if_exists=False,
)
report


## 5. Review the remap report and output files


In [ ]:
report_on_disk = json.loads(REPORT_PATH.read_text())
report_df = pd.DataFrame(report_on_disk["cuts"])
display(report_df)

print({
    "n_annotations_total": report_on_disk["n_annotations_total"],
    "n_assigned": report_on_disk["n_assigned"],
    "n_unassigned": report_on_disk["n_unassigned"],
    "unassigned_ids": report_on_disk["unassigned_ids"],
})

for cut in manifest["cuts"]:
    ann_path = CUTS_DIR / f"{cut["name"]}_annotations.geojson"
    ann_geojson = json.loads(ann_path.read_text())
    print(cut["name"], len(ann_geojson["features"]))


## 6. Overlay annotations on each cut

The figures below load the lowest-resolution TIFF level for each cut and overlay the cut-local annotations after scaling them to that pyramid level.


In [ ]:
def stage_from_feature(feature: dict) -> int:
    label = feature.get("properties", {}).get("metadata", {}).get("ANNOTATION_DESCRIPTION", "")
    if label and label[-1].isdigit():
        return int(label[-1])
    return 0


def ring_from_feature(feature: dict) -> list[list[float]]:
    geometry = feature["geometry"]
    if geometry["type"] == "Polygon":
        return geometry["coordinates"][0]
    return geometry["coordinates"]


overlay_rows = []
for cut in manifest["cuts"]:
    cut_name = cut["name"]
    tif_path = CUTS_DIR / f"{cut_name}.tif"
    ann_path = CUTS_DIR / f"{cut_name}_annotations.geojson"
    ann_geojson = json.loads(ann_path.read_text())

    with tifffile.TiffFile(tif_path) as tif:
        level_index = len(tif.pages) - 1
        thumb = tif.pages[level_index].asarray()
        level_height, level_width = thumb.shape[:2]

    cut_width = cut["level0_bbox"]["x1"] - cut["level0_bbox"]["x0"]
    cut_height = cut["level0_bbox"]["y1"] - cut["level0_bbox"]["y0"]
    scale_x = level_width / cut_width
    scale_y = level_height / cut_height

    fig, ax = plt.subplots(figsize=(12, 5))
    ax.imshow(thumb, cmap="gray")
    for feature in ann_geojson["features"]:
        ring = ring_from_feature(feature)
        xs = [coord[0] * scale_x for coord in ring]
        ys = [coord[1] * scale_y for coord in ring]
        stage = stage_from_feature(feature)
        colour = STAGE_COLOURS.get(stage, "white")
        ax.plot(xs + [xs[0]], ys + [ys[0]], color=colour, linewidth=0.8)

    ax.set_title(f"{cut_name} | n_annotations={len(ann_geojson['features'])}")
    ax.axis("off")
    plt.tight_layout()
    plt.show()

    overlay_rows.append(
        {
            "cut_name": cut_name,
            "level_width": level_width,
            "level_height": level_height,
            "scale_x": round(scale_x, 6),
            "scale_y": round(scale_y, 6),
            "n_annotations": len(ann_geojson["features"]),
        }
    )

display(pd.DataFrame(overlay_rows))


## 7. Manual checks

For each overlay figure, confirm that the outlines stay on tissue, remain within the cut image bounds, and preserve the expected stage colour mapping.
